# 05 — Train + register 4 models to Unity Catalog

RF and GBT classifiers, each in a **pre-departure** (no `dep_delay`) and **in-flight**
(with `dep_delay`) variant. Hyperopt TPE with bounded search — SparkML on serverless
caps model size at 100 MB. All runs log to MLflow. Champions are registered under
3-level UC names with the `@champion` alias.

**Exit test** (Phase 2 of MIGRATION_PLAN.md): re-open the model by alias from a fresh
session and score 10 rows. That's the money shot.

## Environment check
**Environment version 4** is required for `pyspark.ml` and `mlflow.spark` on serverless.

In [0]:
%pip install hyperopt -q
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import sys
sys.path.append("..")

# Core imports
import mlflow
import pyspark.ml
import pyspark.sql.functions as F
from src import config

# PySpark ML imports
from pyspark.ml.feature import VectorAssembler, VectorSlicer
from pyspark.ml.functions import vector_to_array
from pyspark.ml.classification import GBTClassifier, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# MLflow imports
from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature

# Hyperopt imports
from hyperopt import fmin, hp, tpe, Trials, STATUS_OK

# PySpark SQL imports
from pyspark.sql.functions import col

print(f"Python:   {sys.version.split()[0]}")
print(f"MLflow:   {mlflow.__version__}")
print(f"PySpark:  {pyspark.__version__}")

Python:   3.12.3
MLflow:   3.8.1
PySpark:  4.1.0


## MLflow registry setup — the actual Phase 2 fix
The original project set the workspace registry (`databricks`) at train time and read
from Unity Catalog (`databricks-uc`) at score time — an unresolvable name mismatch.

In [0]:
mlflow.set_registry_uri(config.MLFLOW_REGISTRY_URI)
mlflow.set_experiment(config.MLFLOW_EXPERIMENT)
print(f"Registry: {config.MLFLOW_REGISTRY_URI}")
print(f"Experiment: {config.MLFLOW_EXPERIMENT}")

Registry: databricks-uc
Experiment: /Shared/flight-delay-platform


## Load Gold, split

In [0]:
gold = spark.table(config.GOLD).select("features", "label", "dep_delay")
train, test = gold.randomSplit(
    [config.TRAIN_FRACTION, 1.0 - config.TRAIN_FRACTION],
    seed=config.RANDOM_SEED,
)
print(f"Train: {train.count():,}  Test: {test.count():,}")

Train: 1,972,093  Test: 491,886


## Two feature views — the "no dep_delay" ablation

In [0]:
# `dep_delay` is at position 11 of the numerical block (see 04_gold assembled_cols).
# The pre-departure model must not see it. We slice at position 11 for the pre-departure
# view; both views share the same StandardScaler stats.

# Explode the scaled feature vector once and rebuild two views.
gold_arrays = gold.withColumn("f_arr", vector_to_array(col("features")))
n_features = len(gold_arrays.select("f_arr").first()["f_arr"])

pre_indices = [i for i in range(n_features) if i != 11]
in_indices = list(range(n_features))

In [0]:
def with_view(df, indices, name):
    slicer = VectorSlicer(inputCol="features", outputCol=name, indices=indices)
    return slicer.transform(df).select(F.col(name).alias("features"), "label")

train_pre = with_view(train, pre_indices, "features_pre")
test_pre = with_view(test, pre_indices, "features_pre")
train_in = with_view(train, in_indices, "features_in")
test_in = with_view(test, in_indices, "features_in")

## Training loop
Bounded search space respects the 100 MB serverless SparkML cap.

In [0]:
evaluator = BinaryClassificationEvaluator(metricName="areaUnderROC")

RF_SPACE = {
    "numTrees": hp.choice("numTrees", [30, 40, 50]),
    "maxDepth": hp.choice("maxDepth", [6, 7, 8]),
    "minInstancesPerNode": hp.choice("minInstancesPerNode", [25, 50]),
}
GBT_SPACE = {
    "maxIter": hp.choice("maxIter", [20, 25, 30]),
    "maxDepth": hp.choice("maxDepth", [4, 5, 6]),
    "stepSize": hp.uniform("stepSize", 0.05, 0.15),
}

def _train_rf(params, tr):
    return RandomForestClassifier(
        featuresCol="features", labelCol="label",
        numTrees=int(params["numTrees"]),
        maxDepth=int(params["maxDepth"]),
        minInstancesPerNode=int(params["minInstancesPerNode"]),
        seed=config.RANDOM_SEED,
    ).fit(tr)

def _train_gbt(params, tr):
    return GBTClassifier(
        featuresCol="features", labelCol="label",
        maxIter=int(params["maxIter"]),
        maxDepth=int(params["maxDepth"]),
        stepSize=float(params["stepSize"]),
        seed=config.RANDOM_SEED,
    ).fit(tr)

def _log_metrics(model, tr, te, params, tag):
    train_auc = evaluator.evaluate(model.transform(tr))
    test_auc = evaluator.evaluate(model.transform(te))
    mlflow.log_params({f"{tag}_{k}": v for k, v in params.items()})
    mlflow.log_metric(f"{tag}_train_auc", train_auc)
    mlflow.log_metric(f"{tag}_test_auc", test_auc)
    return test_auc, model

def _search(space, train_fn, tr, te, tag, evals):
    trials = Trials()
    best_state = {"auc": -1.0, "model": None, "params": None}

    def objective(params):
        # Train model
        model = train_fn(params, tr)
        
        # Evaluate on both train and test
        train_auc = evaluator.evaluate(model.transform(tr))
        test_auc = evaluator.evaluate(model.transform(te))
        
        # Log intermediate trial to MLflow
        with mlflow.start_run(run_name=f"{tag}_trial", nested=True):
            mlflow.log_params(params)
            mlflow.log_metric("train_auc", train_auc)
            mlflow.log_metric("test_auc", test_auc)
        
        # Track best model
        if test_auc > best_state["auc"]:
            best_state.update({"auc": test_auc, "model": model, "params": params})
        
        return {"loss": -test_auc, "status": STATUS_OK}

    # Run search within parent MLflow run
    with mlflow.start_run(run_name=f"{tag}_search"):
        fmin(objective, space, algo=tpe.suggest, max_evals=evals, trials=trials,
             rstate=None, show_progressbar=False)
        
        # Log best trial results
        mlflow.log_params({f"best_{k}": v for k, v in best_state["params"].items()})
        mlflow.log_metric("best_test_auc", best_state["auc"])
        
    return best_state

## Register champions to Unity Catalog

In [0]:
def _register_champion(model, view_train_df, uc_name, run_name):
    sample_input = view_train_df.limit(5)
    predictions = model.transform(sample_input)
    signature = infer_signature(sample_input, predictions)
    with mlflow.start_run(run_name=run_name):
        model_info = mlflow.spark.log_model(
            spark_model=model,
            artifact_path="model",
            dfs_tmpdir=config.ARTIFACT_VOLUME,
            registered_model_name=uc_name,
            signature=signature,
        )
    client = MlflowClient()
    latest = model_info.registered_model_version
    client.set_registered_model_alias(uc_name, config.CHAMPION_ALIAS, latest)
    print(f"{uc_name}  v{latest}  aliased @{config.CHAMPION_ALIAS}")

## RF pre-departure

In [0]:
best = _search(RF_SPACE, _train_rf, train_pre, test_pre, "rf_pre", config.HYPEROPT_MAX_EVALS)
print(f"RF pre-departure  test AUC = {best['auc']:.4f}")
_register_champion(best["model"], train_pre, config.MODEL_RF_PRE, "rf_pre_departure")

2026/08/31 17:23:39 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o93.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumented

RF pre-departure  test AUC = 0.6341


2026/08/31 17:40:58 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o157.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumente

Uploading artifacts:   0%|          | 0/24 [00:00<?, ?it/s]

🔗 Created version '3' of model 'workspace.flights.rf_pre_departure': https://dbc-d56728e5-4d72.cloud.databricks.com/explore/data/models/workspace/flights/rf_pre_departure/version/3?o=991140949344710


workspace.flights.rf_pre_departure  v3  aliased @champion


## GBT pre-departure

In [0]:
best = _search(GBT_SPACE, _train_gbt, train_pre, test_pre, "gbt_pre", config.HYPEROPT_MAX_EVALS)
print(f"GBT pre-departure  test AUC = {best['auc']:.4f}")
_register_champion(best["model"], train_pre, config.MODEL_GBT_PRE, "gbt_pre_departure")

2026/08/31 17:41:12 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o180.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumente

GBT pre-departure  test AUC = 0.6581


2026/08/31 19:23:24 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o244.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumente

Uploading artifacts:   0%|          | 0/24 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.flights.gbt_pre_departure': https://dbc-d56728e5-4d72.cloud.databricks.com/explore/data/models/workspace/flights/gbt_pre_departure/version/1?o=991140949344710


workspace.flights.gbt_pre_departure  v1  aliased @champion


## RF in-flight

In [0]:
best = _search(RF_SPACE, _train_rf, train_in, test_in, "rf_in", config.HYPEROPT_MAX_EVALS)
print(f"RF in-flight  test AUC = {best['auc']:.4f}")
_register_champion(best["model"], train_in, config.MODEL_RF_IN, "rf_in_flight")

2026/08/31 19:23:39 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o267.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumente

RF in-flight  test AUC = 0.8730


2026/08/31 19:42:52 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o331.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumente

Uploading artifacts:   0%|          | 0/24 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.flights.rf_in_flight': https://dbc-d56728e5-4d72.cloud.databricks.com/explore/data/models/workspace/flights/rf_in_flight/version/1?o=991140949344710


workspace.flights.rf_in_flight  v1  aliased @champion


## GBT in-flight

In [0]:
best = _search(GBT_SPACE, _train_gbt, train_in, test_in, "gbt_in", config.HYPEROPT_MAX_EVALS)
print(f"GBT in-flight  test AUC = {best['auc']:.4f}")
_register_champion(best["model"], train_in, config.MODEL_GBT_IN, "gbt_in_flight")

2026/08/31 19:43:08 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o354.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumente

GBT in-flight  test AUC = 0.9315


2026/08/31 21:56:02 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o418.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumente

Uploading artifacts:   0%|          | 0/24 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.flights.gbt_in_flight': https://dbc-d56728e5-4d72.cloud.databricks.com/explore/data/models/workspace/flights/gbt_in_flight/version/1?o=991140949344710


workspace.flights.gbt_in_flight  v1  aliased @champion


## Phase 2 exit test — load champion by alias, score 10 rows
If this cell prints predictions, the original defect is dead. Screenshot it.

In [0]:
reloaded = mlflow.spark.load_model(f"models:/{config.MODEL_GBT_PRE}@{config.CHAMPION_ALIAS}", dfs_tmpdir=config.ARTIFACT_VOLUME)
reloaded.transform(test_pre.limit(10)).select("label", "prediction", "probability").show(truncate=False)

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|0.0  |0.0       |[0.6218728746036692,0.37812712539633075]|
|0.0  |0.0       |[0.7274138871093284,0.2725861128906716] |
|0.0  |0.0       |[0.849083266735838,0.150916733264162]   |
|0.0  |0.0       |[0.8653645924577333,0.13463540754226666]|
|0.0  |0.0       |[0.8728228688927868,0.12717713110721318]|
|0.0  |0.0       |[0.8096081422305231,0.19039185776947687]|
|1.0  |0.0       |[0.5557153656742405,0.4442846343257595] |
|0.0  |0.0       |[0.908744879502047,0.09125512049795304] |
|0.0  |0.0       |[0.6174723607844174,0.38252763921558264]|
|1.0  |0.0       |[0.6195952247129758,0.3804047752870242] |
+-----+----------+----------------------------------------+

